# Câu a. Thiết kế và sinh dữ liệu khách hàng thương mại điện tử

**Mã sinh viên:** s = 24022476

| Tham số | Phép tính | Giá trị | Yêu cầu | Thực hiện |
|---|---|---|---|---|
| a = s mod 5 | 24022476 mod 5 | 1 | N = 200 + 20a = **220** bản ghi | 220 |
| b = (s div 10) mod 5 | 2402247 mod 5 | 2 | ≥ 8 + b = **10** thuộc tính đầu vào | 13 |
| c = (s div 100) mod 5 | 240224 mod 5 | 4 | ≥ 5 + 2c = **13** bản ghi ngoại lai | 15 |
| Giá trị thiếu | | | ≥ 2 thuộc tính | 3 |
| Nhiễu | | | ≥ 2 thuộc tính số | 3 |

In [1]:
s = 24022476
a, b, c = s % 5, (s // 10) % 5, (s // 100) % 5
N, MIN_ATTRS, MIN_OUTLIERS = 200 + 20 * a, 8 + b, 5 + 2 * c
print(f"a={a}, b={b}, c={c}  ->  N={N}, so thuoc tinh >= {MIN_ATTRS}, so ngoai lai >= {MIN_OUTLIERS}")

a=1, b=2, c=4  ->  N=220, so thuoc tinh >= 10, so ngoai lai >= 13


## 1. Business Understanding (CRISP-DM pha 1)

**Mục tiêu kinh doanh.** Sàn TMĐT muốn chia khách hàng thành các nhóm có hành vi mua sắm khác nhau để (i) thiết kế chương trình tiếp thị riêng, (ii) ưu tiên chăm sóc nhóm giá trị cao, (iii) nhận diện nhóm sắp rời bỏ.

**Tiêu chí thành công** (đặt trước khi có kết quả, dùng lại ở câu e):

- Kinh doanh: mỗi cụm diễn giải được bằng ngôn ngữ nghiệp vụ và có hành động tiếp thị riêng.
- Kỹ thuật: 3–6 cụm; mỗi cụm ≥ 5% số khách; ARI so với phân khúc ẩn ≥ 0,6; silhouette ≥ 0,25.

## 2. Thiết kế thuộc tính (data dictionary)

Khung thiết kế: RFM (Recency, Frequency, Monetary) + nhân khẩu học + hành vi tương tác. Phân loại kiểu theo giáo trình *Data understanding*.

| # | Thuộc tính | Kiểu | Đơn vị / miền | Thống kê hợp lệ | Ý nghĩa và lý do chọn |
|---|---|---|---|---|---|
| 1 | age | số, rời rạc, ratio | 18–70 | mean, median, mode | Tuổi: liên quan khả năng chi tiêu, kênh tiếp thị |
| 2 | gender | nhị phân đối xứng | Nam / Nữ | mode | Giữ làm **phép thử**: sinh độc lập với phân khúc, để kiểm tra mô hình có bị một thuộc tính không mang cấu trúc cụm làm nhiễu hay không |
| 3 | region | nominal | Miền Bắc / Trung / Nam | mode | Logistics, khuyến mãi theo vùng |
| 4 | membership_tier | ordinal | Bronze < Silver < Gold < Platinum | median, mode | Hạng thành viên, rời rạc hóa từ tổng chi tiêu |
| 5 | tenure_months | số, rời rạc, ratio | tháng | mean, median, mode | Thời gian gắn bó |
| 6 | annual_income | số, liên tục, ratio | triệu VND/năm | mean, median | Khả năng chi trả |
| 7 | orders_per_year | số, rời rạc, ratio | đơn/năm | mean, median, mode | F trong RFM |
| 8 | avg_order_value | số, liên tục, ratio | nghìn VND/đơn | mean, median | M trong RFM |
| 9 | days_since_last_purchase | số, rời rạc, ratio | 0–365 ngày | mean, median | R trong RFM, nhận diện khách sắp rời bỏ |
| 10 | return_rate | số, liên tục, ratio | [0, 1] | mean, median | Tỷ lệ hoàn hàng: chi phí vận hành |
| 11 | discount_usage_rate | số, liên tục, ratio | [0, 1] | mean, median | Độ nhạy khuyến mãi |
| 12 | sessions_per_month | số, rời rạc, ratio | phiên/tháng | mean, median, mode | Mức tương tác với app |
| 13 | preferred_category | nominal | 5 ngành hàng | mode | Gợi ý sản phẩm |

Cột phụ, **không phải thuộc tính đầu vào**: `customer_id` (định danh) và `true_segment` (phân khúc ẩn dùng để sinh dữ liệu, chỉ dùng đánh giá ARI/NMI ở câu e).

## 3. Cơ chế sinh dữ liệu sạch

Dữ liệu sinh theo **4 phân khúc ẩn** (hỗn hợp phân phối), mỗi phân khúc có bộ tham số riêng. Trong từng phân khúc:

| Thuộc tính | Phân phối | Lý do |
|---|---|---|
| annual_income | Log-normal | Luôn dương, lệch phải (mean > median) |
| avg_order_value | Log-normal, log(AOV) = log(med) + 0,5·(log(income) − log(med_income)) + ε | Lệch phải; quan hệ tuyến tính trong không gian log |
| orders_per_year | Negative Binomial (≥ 1) | Biến đếm quá phân tán, Poisson quá chặt |
| days_since_last_purchase | Exponential(365/orders) × LogNormal(0; 0,5) + thành phần "ngừng mua" U(60,180) với xác suất riêng từng phân khúc | Thời gian chờ của quá trình mua; **nhiễu riêng** để R không dư thừa hoàn toàn với F và có nhóm "mua đều nhưng đã ngừng" |
| sessions_per_month | Negative Binomial, mean = 3 + 0,7·orders | Cặp thuộc tính dư thừa có chủ đích |
| return_rate, discount_usage_rate | Beta | Bị chặn tự nhiên trong [0, 1] |
| age | Normal làm tròn, chặn [18, 70] | Gần đối xứng trong nhóm, đa đỉnh khi gộp |
| tenure_months | Gamma, chặn tenure ≤ (age − 16)·12 | Dương, lệch phải; có ràng buộc logic |
| membership_tier | Rời rạc hóa orders × AOV theo ngưỡng 5.000 / 14.000 / 40.000 (nghìn VND/năm), 10% lệch ±1 bậc | Ordinal sinh từ thuộc tính số |
| preferred_category | Categorical, xác suất phụ thuộc phân khúc | Có liên hệ với phân khúc |
| region, gender | Categorical / Bernoulli(0,5), **độc lập** với phân khúc | Thuộc tính không mang cấu trúc cụm |

In [2]:
import numpy as np, pandas as pd
from scipy import stats
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, REPORTS = ROOT / "data" / "raw", ROOT / "reports"
RAW.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)

SEED = 2476                      # 4 chu so cuoi cua ma sinh vien
rng = np.random.default_rng(SEED)

SEGMENTS = {
    # n, tuoi, thu nhap (median, sigma log), AOV (median, sigma log), so don (mean, k), xs ngung mua,
    # tenure (mean, shape), Beta hoan hang, Beta khuyen mai, xs nganh hang
    "VIP":         dict(n=30, age=(42, 7), inc=(600, 0.30), aov=(1800, 0.30), ord=(30, 6), lapse=0.05,
                        ten=(48, 3), ret=(2, 25), disc=(2, 8),  cat=[.45, .15, .25, .10, .05]),
    "DealHunter":  dict(n=70, age=(26, 5), inc=(150, 0.30), aov=(220, 0.30),  ord=(45, 6), lapse=0.10,
                        ten=(24, 3), ret=(3, 15), disc=(7, 3),  cat=[.10, .40, .10, .25, .15]),
    "Loyal":       dict(n=60, age=(36, 8), inc=(280, 0.30), aov=(550, 0.30),  ord=(24, 6), lapse=0.10,
                        ten=(60, 4), ret=(2, 22), disc=(4, 6),  cat=[.15, .15, .30, .10, .30]),
    "NewInactive": dict(n=60, age=(29, 8), inc=(180, 0.35), aov=(400, 0.35),  ord=(4, 2),  lapse=0.35,
                        ten=(5, 2),  ret=(2, 18), disc=(3, 5),  cat=[.20, .30, .15, .20, .15]),
}
assert sum(p["n"] for p in SEGMENTS.values()) == N

CATS    = ["Điện tử", "Thời trang", "Gia dụng", "Mỹ phẩm", "Thực phẩm"]
REGIONS = ["Miền Bắc", "Miền Trung", "Miền Nam"]
TIERS   = ["Bronze", "Silver", "Gold", "Platinum"]
TIER_CUTS = [5_000, 14_000, 40_000]   # nghin VND / nam
BETA_INC = 0.5                        # do co gian cua AOV theo thu nhap (trong khong gian log)

def nb(mean, k, size=None):
    # Negative Binomial tham so hoa theo mean va do phan tan k (var = mean + mean^2/k)
    return rng.negative_binomial(k, k / (k + mean), size)

def gen_segment(name, p):
    n = p["n"]
    age = np.clip(np.round(rng.normal(*p["age"], n)), 18, 70).astype(int)
    income = rng.lognormal(np.log(p["inc"][0]), p["inc"][1], n)
    aov = np.exp(np.log(p["aov"][0]) + BETA_INC * (np.log(income) - np.log(p["inc"][0]))
                 + rng.normal(0, p["aov"][1], n))
    orders = np.maximum(nb(p["ord"][0], p["ord"][1], n), 1)
    sessions = nb(3 + 0.7 * orders, 20)
    # Recency: thoi gian cho ~ Exp(365/orders), nhan nhieu rieng, cong thanh phan "ngung mua"
    recency = rng.exponential(365 / orders) * rng.lognormal(0, 0.5, n)
    recency += (rng.random(n) < p["lapse"]) * rng.uniform(60, 180, n)
    recency = np.clip(np.round(recency), 0, 365).astype(int)
    tenure = rng.gamma(p["ten"][1], p["ten"][0] / p["ten"][1], n)
    tenure = np.clip(np.round(tenure), 1, (age - 16) * 12).astype(int)
    tier = np.digitize(orders * aov, TIER_CUTS)
    flip = rng.random(n) < 0.10
    tier = np.clip(tier + flip * rng.choice([-1, 1], n), 0, 3)
    return pd.DataFrame({
        "age": age,
        "gender": rng.choice(["Nam", "Nữ"], n),
        "region": rng.choice(REGIONS, n, p=[.35, .20, .45]),
        "membership_tier": np.array(TIERS)[tier],
        "tenure_months": tenure,
        "annual_income": np.round(income, 1),
        "orders_per_year": orders,
        "avg_order_value": np.round(aov, 1),
        "days_since_last_purchase": recency,
        "return_rate": np.round(rng.beta(*p["ret"], n), 3),
        "discount_usage_rate": np.round(rng.beta(*p["disc"], n), 3),
        "sessions_per_month": sessions,
        "preferred_category": rng.choice(CATS, n, p=p["cat"]),
        "true_segment": name,
    })

clean = pd.concat([gen_segment(k, p) for k, p in SEGMENTS.items()], ignore_index=True)
clean = clean.sample(frac=1, random_state=SEED).reset_index(drop=True)   # tron thu tu dong
clean.insert(0, "customer_id", [f"C{i:04d}" for i in range(1, N + 1)])

INPUT_COLS = [c_ for c_ in clean.columns if c_ not in ("customer_id", "true_segment")]
NUM_COLS = ["age", "tenure_months", "annual_income", "orders_per_year", "avg_order_value",
            "days_since_last_purchase", "return_rate", "discount_usage_rate", "sessions_per_month"]
print(clean.shape, "| so thuoc tinh dau vao:", len(INPUT_COLS))
clean.head()

(220, 15) | so thuoc tinh dau vao: 13


,customer_id,age,gender,region,membership_tier,tenure_months,annual_income,orders_per_year,avg_order_value,days_since_last_purchase,return_rate,discount_usage_rate,sessions_per_month,preferred_category,true_segment
0,C0001,32,Nữ,Miền Trung,Bronze,2,91.8,1,475.7,197,0.022,0.230,4,Mỹ phẩm,NewInactive
1,C0002,41,Nam,Miền Nam,Silver,33,320.0,17,680.6,56,0.168,0.322,14,Mỹ phẩm,Loyal
2,C0003,51,Nam,Miền Bắc,Gold,48,692.1,8,2131.8,5,0.067,0.130,9,Mỹ phẩm,VIP
3,C0004,33,Nữ,Miền Trung,Gold,54,555.9,18,1328.3,92,0.275,0.211,16,Điện tử,Loyal
4,C0005,37,Nam,Miền Trung,Gold,89,451.7,25,1333.5,5,0.052,0.131,19,Gia dụng,VIP


In [3]:
def check_constraints(df):
    # Rang buoc mien gia tri va logic; tra ve so dong vi pham tung rang buoc
    return pd.Series({
        "age ngoai [18,70]": (~df.age.between(18, 70)).sum(),
        "ty le ngoai [0,1]": (~df.return_rate.between(0, 1) | ~df.discount_usage_rate.between(0, 1)).sum(),
        "AOV <= 0": (df.avg_order_value <= 0).sum(),
        "income <= 0": (df.annual_income <= 0).sum(),
        "orders < 1": (df.orders_per_year < 1).sum(),
        "sessions < 0": (df.sessions_per_month < 0).sum(),
        "recency ngoai [0,365]": (~df.days_since_last_purchase.between(0, 365)).sum(),
        "tenure > (age-16)*12": (df.tenure_months > (df.age - 16) * 12).sum(),
    })

assert check_constraints(clean).sum() == 0 and clean.isna().sum().sum() == 0
print("Du lieu sach: khong vi pham rang buoc, khong co gia tri thieu.")
clean.groupby("true_segment")[NUM_COLS].median().round(2).T

Du lieu sach: khong vi pham rang buoc, khong co gia tri thieu.


true_segment,DealHunter,Loyal,NewInactive,VIP
age,25.00,35.50,30.00,44.00
tenure_months,21.00,55.50,3.00,42.00
annual_income,148.30,274.65,185.65,655.20
orders_per_year,43.50,23.00,3.00,24.00
avg_order_value,218.40,528.65,418.85,1699.80
days_since_last_purchase,5.50,16.00,118.00,16.00
return_rate,0.18,0.09,0.08,0.06
discount_usage_rate,0.74,0.41,0.30,0.19
sessions_per_month,31.00,17.50,5.50,18.50


## 4. Làm bẩn dữ liệu

Thứ tự: **nhiễu → ngoại lai → giá trị thiếu**. Ngoại lai đặt sau nhiễu để giá trị ngoại lai đã thiết kế không bị nhiễu làm lệch; giá trị thiếu đặt cuối và không rơi vào các dòng ngoại lai để không làm mất dấu ngoại lai. Mọi thay đổi được ghi vào `injection_log`.

### 4.1. Nhiễu ngẫu nhiên (3 thuộc tính số, toàn bộ 220 dòng)

- `avg_order_value` × (1 + N(0; 0,08)) và `annual_income` × (1 + N(0; 0,05)): nhiễu **nhân**, vì hai biến log-normal có độ lớn chênh nhau nhiều lần giữa các khách; nhiễu cộng sẽ quá lớn với khách chi tiêu thấp và có thể tạo giá trị âm.
- `sessions_per_month` + N(0; 2), làm tròn, chặn ≥ 0: nhiễu **cộng** cho biến đếm.

In [4]:
dirty = clean.copy()
log = []
def record(idx, col, kind, old, new, note=""):
    log.append(dict(customer_id=dirty.at[idx, "customer_id"], column=col, kind=kind,
                    old_value=old, new_value=new, note=note))

noise_spec = {
    "avg_order_value":    lambda x: np.round(x * (1 + rng.normal(0, 0.08, len(x))), 1),
    "annual_income":      lambda x: np.round(x * (1 + rng.normal(0, 0.05, len(x))), 1),
    "sessions_per_month": lambda x: np.clip(np.round(x + rng.normal(0, 2, len(x))), 0, None).astype(int),
}
rows = []
for col, f in noise_spec.items():
    new = f(clean[col].to_numpy())
    for i in range(N):
        record(i, col, "noise", clean.at[i, col], new[i])
    dirty[col] = new
    d = new - clean[col]
    rows.append(dict(column=col, noise_sd=d.std(), signal_sd=clean[col].std(),
                     var_ratio=d.var() / clean[col].var(), changed=(d != 0).mean()))
noisy_only = dirty.copy()          # trang thai sau nhieu, truoc ngoai lai (chi de kiem tra)
assert check_constraints(noisy_only).sum() == 0, "Nhieu khong duoc tao gia tri bat hop le"
pd.DataFrame(rows).round(4)

,column,noise_sd,signal_sd,var_ratio,changed
0,avg_order_value,62.8169,546.4342,0.0132,0.9864
1,annual_income,15.2822,175.9905,0.0075,0.9864
2,sessions_per_month,2.1072,15.2242,0.0192,0.8182


### 4.2. Ngoại lai (15 bản ghi, sửa trên dòng có sẵn để giữ N = 220)

| Loại | Số bản ghi | Cách tạo | Kỳ vọng phát hiện |
|---|---|---|---|
| `outlier_error` (lỗi đơn biến) | 5 | age = 150; age = 3; AOV = −450; return_rate = 2,5; sessions = 900 | Boxplot/IQR, kiểm tra miền giá trị |
| `outlier_logic` (lỗi logic đa biến) | 3 | tuổi ≤ 22 nhưng tenure = 96 tháng; orders = 80 nhưng recency = 180 ngày; khách mới chi tiêu rất thấp nhưng hạng Platinum | Từng giá trị nằm trong khoảng bình thường, IQR đơn biến không bắt được; cần kiểm tra ràng buộc |
| `outlier_valid` (hợp lệ) | 7 | 4 khách sỉ (200–350 đơn/năm, AOV 3.000–6.000) và 3 siêu VIP (AOV 25.000–40.000, thu nhập 3.000–5.000) | IQR bắt được nhưng là khách thật có giá trị cao, không được mặc định xóa |

Ngoại lai lỗi lấy từ các dòng ngoài phân khúc VIP; ngoại lai hợp lệ lấy từ các dòng VIP (bản chất vẫn là khách giá trị cao nên `true_segment` giữ nguyên).

In [5]:
used = set()
def pick(mask, k=1):
    idx = rng.choice(dirty.index[mask & ~dirty.index.isin(list(used))], k, replace=False)
    used.update(idx.tolist())
    return idx.tolist()

def set_val(i, col, new, kind, note):
    record(i, col, kind, dirty.at[i, col], new, note)
    dirty[col] = dirty[col].astype(object) if isinstance(new, float) and dirty[col].dtype.kind == "i" else dirty[col]
    dirty.at[i, col] = new

non_vip = dirty.true_segment != "VIP"

# (1) Loi don bien: 5 ban ghi
e = pick(non_vip, 5)
set_val(e[0], "age", 150, "outlier_error", "tuoi bat kha")
set_val(e[1], "age", 3, "outlier_error", "tuoi bat kha")
set_val(e[2], "avg_order_value", -450.0, "outlier_error", "gia tri don am")
set_val(e[3], "return_rate", 2.5, "outlier_error", "ty le > 1")
set_val(e[4], "sessions_per_month", 900, "outlier_error", "so phien bat kha")

# (2) Loi logic da bien: 3 ban ghi
i = pick(non_vip & (dirty.age <= 22))[0]
set_val(i, "tenure_months", 96, "outlier_logic", "tenure > (age-16)*12")
i = pick(dirty.true_segment == "DealHunter")[0]
set_val(i, "orders_per_year", 80, "outlier_logic", "mua 80 don/nam nhung 180 ngay chua mua")
set_val(i, "days_since_last_purchase", 180, "outlier_logic", "mua 80 don/nam nhung 180 ngay chua mua")
i = pick((dirty.true_segment == "NewInactive") & (dirty.membership_tier == "Bronze")
         & (dirty.orders_per_year * dirty.avg_order_value < 2000))[0]
set_val(i, "membership_tier", "Platinum", "outlier_logic", "chi tieu < 2 trieu/nam nhung hang Platinum")

# (3) Hop le: 4 khach si + 3 sieu VIP, lay tu phan khuc VIP
v = pick(dirty.true_segment == "VIP", 7)
for i in v[:4]:
    set_val(i, "orders_per_year", int(rng.integers(200, 351)), "outlier_valid", "khach si")
    set_val(i, "avg_order_value", float(np.round(rng.uniform(3000, 6000), 1)), "outlier_valid", "khach si")
    set_val(i, "sessions_per_month", int(rng.integers(120, 201)), "outlier_valid", "khach si")
    set_val(i, "days_since_last_purchase", int(rng.integers(0, 4)), "outlier_valid", "khach si")
    set_val(i, "membership_tier", "Platinum", "outlier_valid", "khach si")
for i in v[4:]:
    set_val(i, "avg_order_value", float(np.round(rng.uniform(25000, 40000), 1)), "outlier_valid", "sieu VIP")
    set_val(i, "annual_income", float(np.round(rng.uniform(3000, 5000), 1)), "outlier_valid", "sieu VIP")
    set_val(i, "membership_tier", "Platinum", "outlier_valid", "sieu VIP")

outlier_rows = sorted(used)
print("So ban ghi ngoai lai:", len(outlier_rows))
dirty.loc[outlier_rows, ["customer_id", "true_segment"] + INPUT_COLS]

So ban ghi ngoai lai: 15


,customer_id,true_segment,age,gender,region,membership_tier,tenure_months,annual_income,orders_per_year,avg_order_value,days_since_last_purchase,return_rate,discount_usage_rate,sessions_per_month,preferred_category
17,C0018,VIP,48,Nữ,Miền Trung,Platinum,33,660.3,258,4908.3,2,0.185,0.221,126,Thời trang
51,C0052,NewInactive,26,Nữ,Miền Nam,Platinum,2,172.0,1,320.2,312,0.190,0.246,2,Thời trang
66,C0067,VIP,51,Nam,Miền Bắc,Platinum,89,709.9,315,3367.1,0,0.089,0.118,133,Điện tử
71,C0072,VIP,42,Nữ,Miền Nam,Platinum,7,3752.0,7,30532.0,20,0.121,0.190,10,Mỹ phẩm
72,C0073,VIP,44,Nam,Miền Bắc,Platinum,32,4428.8,10,35875.8,64,0.034,0.188,1,Gia dụng
110,C0111,Loyal,38,Nữ,Miền Nam,Gold,115,263.2,30,482.0,4,2.500,0.293,11,Điện tử
121,C0122,VIP,32,Nữ,Miền Nam,Platinum,120,4967.0,37,34180.9,10,0.038,0.376,27,Điện tử
140,C0141,VIP,46,Nữ,Miền Bắc,Platinum,76,698.9,236,5053.5,0,0.022,0.320,198,Điện tử
166,C0167,NewInactive,43,Nam,Miền Trung,Bronze,11,180.7,4,420.4,196,0.001,0.215,900,Thời trang
167,C0168,Loyal,150,Nam,Miền Nam,Silver,107,171.5,14,328.5,6,0.067,0.526,11,Thực phẩm


### 4.3. Giá trị thiếu (3 thuộc tính)

| Thuộc tính | Cơ chế | Số ô | Cách tạo |
|---|---|---|---|
| age | MCAR | 11 (5%) | Chọn ngẫu nhiên đều |
| annual_income | **MAR** | 22 (10%) | Xác suất thiếu ∝ exp(−tenure/12) + 0,05: khách mới chưa khai báo hồ sơ nên thiếu nhiều hơn. Phụ thuộc vào `tenure_months` là biến quan sát được |
| preferred_category | MCAR | 9 (4%) | Chọn ngẫu nhiên đều |

Các dòng ngoại lai không bị gán thiếu.

In [6]:
eligible = dirty.index[~dirty.index.isin(outlier_rows)]

def make_missing(col, k, weights=None, note=""):
    p = None if weights is None else (weights / weights.sum()).to_numpy()
    idx = rng.choice(eligible, k, replace=False, p=p)
    for i in idx:
        record(i, col, "missing", dirty.at[i, col], np.nan, note)
    dirty.loc[idx, col] = np.nan

make_missing("age", round(0.05 * N), note="MCAR")
w = np.exp(-dirty.loc[eligible, "tenure_months"].astype(float) / 12) + 0.05
make_missing("annual_income", round(0.10 * N), w, note="MAR theo tenure_months")
make_missing("preferred_category", round(0.04 * N), note="MCAR")

for col in ["age", "tenure_months", "orders_per_year", "days_since_last_purchase", "sessions_per_month"]:
    dirty[col] = pd.to_numeric(dirty[col]).astype("Int64")       # so nguyen cho phep NaN
for col in ["annual_income", "avg_order_value", "return_rate", "discount_usage_rate"]:
    dirty[col] = pd.to_numeric(dirty[col]).astype(float)

log_df = pd.DataFrame(log)
dirty.isna().sum()[lambda s: s > 0]

age                   11
annual_income         22
preferred_category     9
dtype: int64

## 5. Lưu dữ liệu và kiểm tra tự động

In [7]:
clean.to_csv(RAW / "customers_clean.csv", index=False, encoding="utf-8-sig")
dirty.to_csv(RAW / "customers_dirty.csv", index=False, encoding="utf-8-sig")
log_df.to_csv(RAW / "injection_log.csv", index=False, encoding="utf-8-sig")

out_kinds = ["outlier_error", "outlier_logic", "outlier_valid"]
n_out = log_df[log_df.kind.isin(out_kinds)].groupby("kind").customer_id.nunique()
miss_cols = dirty.columns[dirty.isna().any()].tolist()
noise_cols = log_df[log_df.kind == "noise"].column.unique().tolist()

assert len(clean) == len(dirty) == N == 220
assert len(INPUT_COLS) >= MIN_ATTRS
assert len(miss_cols) >= 2
assert len(noise_cols) >= 2 and set(noise_cols) <= set(NUM_COLS)
assert n_out.sum() >= MIN_OUTLIERS
assert (clean.customer_id == dirty.customer_id).all()
# Doc lai file de chac chan luu dung
assert pd.read_csv(RAW / "customers_dirty.csv").shape == dirty.shape

print(f"So ban ghi: {N} | thuoc tinh dau vao: {len(INPUT_COLS)} (yeu cau >= {MIN_ATTRS})")
print(f"Cot co gia tri thieu: {miss_cols}")
print(f"Cot co nhieu: {noise_cols}")
print(f"Ban ghi ngoai lai: {n_out.to_dict()} -> tong {n_out.sum()} (yeu cau >= {MIN_OUTLIERS})")
log_df.groupby(["kind", "column"]).size().unstack(fill_value=0)

So ban ghi: 220 | thuoc tinh dau vao: 13 (yeu cau >= 10)
Cot co gia tri thieu: ['age', 'annual_income', 'preferred_category']
Cot co nhieu: ['avg_order_value', 'annual_income', 'sessions_per_month']
Ban ghi ngoai lai: {'outlier_error': 5, 'outlier_logic': 3, 'outlier_valid': 7} -> tong 15 (yeu cau >= 13)


column,age,annual_income,avg_order_value,days_since_last_purchase,membership_tier,orders_per_year,preferred_category,return_rate,sessions_per_month,tenure_months
kind,,,,,,,,,,
missing,11,22,0,0,0,0,9,0,0,0
noise,0,220,220,0,0,0,0,0,220,0
outlier_error,2,0,1,0,0,0,0,1,1,0
outlier_logic,0,0,0,1,1,1,0,0,0,1
outlier_valid,0,3,7,4,7,4,0,0,4,0


### Kiểm tra thiết kế ngoại lai bằng quy tắc 1,5 × IQR

Kiểm tra nhanh xem thiết kế có đúng ý đồ không (phân tích đầy đủ thuộc câu b): ngoại lai lỗi đơn biến và ngoại lai hợp lệ phải vượt ngưỡng IQR ở ít nhất một cột; ngoại lai logic thì không. Với dòng ngoại lai chỉ xét các cột số đã bị can thiệp; với dòng bình thường xét mọi cột số (tỷ lệ này cho biết IQR gắn cờ nhầm bao nhiêu trên dữ liệu lệch).

In [8]:
def iqr_flags(df, cols):
    f = pd.DataFrame(False, index=df.index, columns=cols)
    for col in cols:
        x = df[col].astype(float)
        q1, q3 = x.quantile([.25, .75]); r = q3 - q1
        f[col] = (x < q1 - 1.5 * r) | (x > q3 + 1.5 * r)
    return f

flags = iqr_flags(dirty, NUM_COLS)
olog = log_df[log_df.kind.isin(out_kinds)]
kind_of = olog.drop_duplicates("customer_id").set_index("customer_id").kind
touched = olog[olog.column.isin(NUM_COLS)].groupby("customer_id").column.apply(list)
def flagged(i):
    # dong ngoai lai: chi xet cac cot so da bi can thiep; dong binh thuong: xet moi cot so
    cid = dirty.at[i, "customer_id"]
    if cid in kind_of.index:
        return bool(flags.loc[i, touched.get(cid, [])].any())
    return bool(flags.loc[i].any())
chk = pd.DataFrame({"kind": dirty.customer_id.map(kind_of).fillna("binh thuong"),
                    "iqr_flag": [flagged(i) for i in dirty.index]})
iqr_summary = chk.groupby("kind").iqr_flag.agg(so_dong="size", bi_iqr_bat="sum")
iqr_summary["ty_le"] = (iqr_summary.bi_iqr_bat / iqr_summary.so_dong).round(2)
assert iqr_summary.loc["outlier_logic", "bi_iqr_bat"] == 0, "Ngoai lai logic phai nam trong nguong IQR"
assert iqr_summary.loc["outlier_error", "ty_le"] == 1 and iqr_summary.loc["outlier_valid", "ty_le"] == 1
iqr_summary

,so_dong,bi_iqr_bat,ty_le
kind,,,
binh thuong,205,50,0.24
outlier_error,5,5,1.00
outlier_logic,3,0,0.00
outlier_valid,7,7,1.00


## 6. Các phát hiện kỳ vọng (để kiểm chứng ở câu b)

Các đặc điểm được cài sẵn vào cơ chế sinh, đo trên **dữ liệu sạch**. Câu b phân tích dữ liệu bẩn mà không nhìn vào đây, sau đó đối chiếu: đặc điểm nào bị nhiễu, ngoại lai, giá trị thiếu làm lệch là bằng chứng định lượng về ảnh hưởng của dữ liệu bẩn.

In [9]:
def corr2(df, x, y, logx=False, logy=False):
    d = df[[x, y]].astype(float).dropna()
    xv = np.log(d[x]) if logx else d[x]; yv = np.log(d[y]) if logy else d[y]
    return stats.pearsonr(xv, yv)[0], stats.spearmanr(d[x], d[y])[0]

def chi2(df, x, y):
    t = pd.crosstab(df[x], df[y]); r = stats.chi2_contingency(t)
    return r[0], r[2], r[1]

F = {}
for col in ["annual_income", "avg_order_value"]:
    F[col] = (clean[col].mean(), clean[col].median(), stats.skew(clean[col]))
F["inc_aov_raw"], F["inc_aov_sp"] = corr2(clean, "annual_income", "avg_order_value")
F["inc_aov_log"] = corr2(clean, "annual_income", "avg_order_value", True, True)[0]
F["ord_rec"] = corr2(clean, "orders_per_year", "days_since_last_purchase")
F["ord_ses"] = corr2(clean, "orders_per_year", "sessions_per_month")
F["chi_cat"] = chi2(clean, "preferred_category", "true_segment")
F["chi_gen"] = chi2(clean, "gender", "true_segment")
F["chi_reg"] = chi2(clean, "region", "true_segment")
m = dirty.annual_income.isna()
F["ten_miss"], F["ten_obs"] = dirty.tenure_months[m].median(), dirty.tenure_months[~m].median()
F["mw_p"] = stats.mannwhitneyu(dirty.tenure_months[m].astype(float), dirty.tenure_months[~m].astype(float)).pvalue
F["miss_by_seg"] = m.groupby(dirty.true_segment).mean().round(3).to_dict()
# Tac dong cua du lieu ban len tuong quan (xem truoc cho cau b)
F["inc_aov_dirty"] = corr2(dirty, "annual_income", "avg_order_value")
F["ord_ses_dirty"] = corr2(dirty, "orders_per_year", "sessions_per_month")

REG_NOTE = ("; p < 0,05 dù sinh độc lập: liên hệ giả do ngẫu nhiên khi N nhỏ, không được diễn giải là có quan hệ thật"
            if F["chi_reg"][2] < 0.05 else "")
md_text = f'''# Các phát hiện kỳ vọng (đo trên dữ liệu sạch, seed = {SEED})

| # | Đặc điểm cài sẵn | Giá trị đo được | Công cụ kiểm tra ở câu b |
|---|---|---|---|
| 1 | annual_income lệch phải | mean {F["annual_income"][0]:.1f} > median {F["annual_income"][1]:.1f}; skewness {F["annual_income"][2]:.2f} | describe, histogram, boxplot |
| 2 | avg_order_value lệch phải | mean {F["avg_order_value"][0]:.1f} > median {F["avg_order_value"][1]:.1f}; skewness {F["avg_order_value"][2]:.2f} | describe, histogram, boxplot |
| 3 | income và AOV tuyến tính trong không gian log | Pearson thang gốc {F["inc_aov_raw"]:.3f}; Pearson log-log {F["inc_aov_log"]:.3f}; Spearman {F["inc_aov_sp"]:.3f} | Pearson trước và sau log, Spearman |
| 4 | orders và recency quan hệ nghịch, phi tuyến | Pearson {F["ord_rec"][0]:.3f}; Spearman {F["ord_rec"][1]:.3f} | So sánh Pearson với Spearman, scatter |
| 5 | orders và sessions dư thừa | Pearson {F["ord_ses"][0]:.3f}; Spearman {F["ord_ses"][1]:.3f} | Ma trận tương quan |
| 6 | preferred_category phụ thuộc phân khúc | chi2 = {F["chi_cat"][0]:.1f}, df = {F["chi_cat"][1]}, p = {F["chi_cat"][2]:.2e} | Kiểm định chi bình phương |
| 7 | gender độc lập với phân khúc | chi2 = {F["chi_gen"][0]:.2f}, df = {F["chi_gen"][1]}, p = {F["chi_gen"][2]:.3f} | Kiểm định chi bình phương |
| 8 | region được sinh độc lập với phân khúc | chi2 = {F["chi_reg"][0]:.2f}, df = {F["chi_reg"][1]}, p = {F["chi_reg"][2]:.3f}{REG_NOTE} | Kiểm định chi bình phương |
| 9 | annual_income thiếu theo cơ chế MAR (đo trên dữ liệu bẩn) | median tenure: nhóm thiếu {F["ten_miss"]} tháng, nhóm không thiếu {F["ten_obs"]} tháng; Mann-Whitney p = {F["mw_p"]:.2e}; tỷ lệ thiếu theo phân khúc {F["miss_by_seg"]} | So sánh phân phối tenure giữa hai nhóm |
| 10 | Ngoại lai logic không bị IQR đơn biến phát hiện | IQR bắt {int(iqr_summary.loc["outlier_logic","bi_iqr_bat"])}/{int(iqr_summary.loc["outlier_logic","so_dong"])} logic; {int(iqr_summary.loc["outlier_error","bi_iqr_bat"])}/{int(iqr_summary.loc["outlier_error","so_dong"])} lỗi đơn biến; {int(iqr_summary.loc["outlier_valid","bi_iqr_bat"])}/{int(iqr_summary.loc["outlier_valid","so_dong"])} hợp lệ; {int(iqr_summary.loc["binh thuong","bi_iqr_bat"])}/{int(iqr_summary.loc["binh thuong","so_dong"])} dòng bình thường bị gắn cờ | IQR so với kiểm tra ràng buộc |

Xem trước tác động của dữ liệu bẩn lên tương quan (Pearson; Spearman):

- income và AOV: sạch ({F["inc_aov_raw"]:.3f}; {F["inc_aov_sp"]:.3f}), bẩn ({F["inc_aov_dirty"][0]:.3f}; {F["inc_aov_dirty"][1]:.3f})
- orders và sessions: sạch ({F["ord_ses"][0]:.3f}; {F["ord_ses"][1]:.3f}), bẩn ({F["ord_ses_dirty"][0]:.3f}; {F["ord_ses_dirty"][1]:.3f})
'''
open(REPORTS / "expected_findings.md", "w", encoding="utf-8").write(md_text)
print(md_text)

# Các phát hiện kỳ vọng (đo trên dữ liệu sạch, seed = 2476)

| # | Đặc điểm cài sẵn | Giá trị đo được | Công cụ kiểm tra ở câu b |
|---|---|---|---|
| 1 | annual_income lệch phải | mean 270.6 > median 210.8; skewness 1.59 | describe, histogram, boxplot |
| 2 | avg_order_value lệch phải | mean 592.2 > median 407.8; skewness 2.27 | describe, histogram, boxplot |
| 3 | income và AOV tuyến tính trong không gian log | Pearson thang gốc 0.827; Pearson log-log 0.820; Spearman 0.760 | Pearson trước và sau log, Spearman |
| 4 | orders và recency quan hệ nghịch, phi tuyến | Pearson -0.510; Spearman -0.661 | So sánh Pearson với Spearman, scatter |
| 5 | orders và sessions dư thừa | Pearson 0.871; Spearman 0.902 | Ma trận tương quan |
| 6 | preferred_category phụ thuộc phân khúc | chi2 = 53.1, df = 12, p = 3.90e-07 | Kiểm định chi bình phương |
| 7 | gender độc lập với phân khúc | chi2 = 0.73, df = 3, p = 0.867 | Kiểm định chi bình phương |
| 8 | region được sinh độc lập với phân khúc | chi2 = 13.

## 7. Giả định và hạn chế

**Giả định**

- Khách hàng thuộc đúng một trong 4 phân khúc ẩn; trong từng phân khúc, phần lớn thuộc tính được sinh độc lập, nên tương quan ở mức tổng thể chủ yếu do biến ẩn `true_segment` tạo ra (bài toán biến thứ ba), không phải quan hệ nhân quả.
- Các bản ghi độc lập và cùng phân phối (IID).
- Dạng phân phối và tham số do người thiết kế chọn theo hiểu biết chung về hành vi mua sắm, không hiệu chỉnh từ dữ liệu thật.

**Hạn chế**

- Các phân khúc có thể tách biệt rõ hơn thực tế, nên kết quả phân cụm trên dữ liệu này lạc quan hơn so với dữ liệu thật.
- Không có yếu tố thời gian (mùa sale, Tết), không có phụ thuộc giữa các khách hàng.
- N = 220 nhỏ: phân khúc VIP chỉ có 30 khách, trong đó 7 khách bị biến thành ngoại lai hợp lệ, nên thống kê theo phân khúc có sai số lớn.
- Ngoại lai và giá trị thiếu được tạo theo kịch bản định trước; dữ liệu thật có thể có các dạng lỗi khác (trùng bản ghi, sai định dạng, sai đơn vị).
- Nhãn `true_segment` của 7 ngoại lai hợp lệ vẫn là VIP; khi tính ARI ở câu e cần nêu rõ có tính các dòng này hay không.